In [ ]:
# load liabries and the dataset
import pandas as pd
import numpy as np
df = pd.read_feather('all_flights_preprocessed1.feather')
display(df)

In [ ]:
# handwritten list of some US holidays, to introduce a new feature
# New Year's Day, Martin Luther King Jr. Day, Presidents' Day, Memorial Day, Independence Day, Labor Day, Columbus Day, Veterans Day, Thanksgiving, Christmas eve Christmas day, new years eve
us_holidays = [
    "2023-01-01",  # New Year's Day
    "2023-01-16",  # Martin Luther King Jr. Day
    "2023-02-20",  # Presidents' Day
    "2023-05-29",  # Memorial Day
    "2023-07-04",  # Independence Day
    "2023-09-04",  # Labor Day
    "2023-10-09",  # Columbus Day
    "2023-11-11",  # Veterans Day
    "2023-11-23",  # Thanksgiving
    "2023-12-24",  # Christmas Eve
    "2023-12-25",  # Christmas Day
    "2023-12-31"   # New Year's Eve
]
# convert the list to datetime
us_holidays = pd.to_datetime(us_holidays)
# introduce a new feature if the flight is on a holiday or not
df['IsHoliday'] = df['CRSDepDateTime'].dt.normalize().isin(us_holidays).astype(int)

# create a feature for the distance to the nearest holiday, by calculating the difference in days between the flight date and the nearest holiday
df['DaysToNearestHoliday'] = df['CRSDepDateTime'].dt.normalize().apply(lambda x: min(abs((x - holiday).days) for holiday in us_holidays))


In [ ]:
# show for each column the number of unique values, missing values and the number of mssing values when rows with Divred and Cannceld flights are dropped, as these flights have a lot of missing values in the arrival delay column
for col in df.columns:
    unique_values = df[col].nunique()
    missing_values = df[col].isna().sum()
    if missing_values > 0:
        print(f"{col}:  Unique:   {unique_values} ---  Missing: {missing_values}")

In [ ]:
# load the airoorts dataset
airports = pd.read_csv('airports_with_runway_info.csv')
# merge the airports dataset with the flights dataset to get the timezone information for the departure and arrival airports
display(airports)
keep_cols = ['iata_code', 'type','scheduled_service','num_runways','most_common_surface','avg_runway_length','has_lighted_runways' ]
weather_cols = ['iata_code']
# get the colum index number of 'airport_station'
airport_station_index = airports.columns.get_loc('airport_station')
# add all columns from the index of 'airport_station' to the end of the dataframe to the list of columns to keep
weather_cols += airports.columns[airport_station_index:].tolist()

In [ ]:
import meteostat as ms
ms.config.block_large_requests = False
airports_weather = airports[weather_cols]
# grop the flights dataset, and search for the fist and last flight date for each airport, and merge this information with the airports dataset
airport_flight_dates = df.groupby('Origin')['knownWeatherDateTime_UTC'].agg(['min', 'max']).reset_index()
# also do the same for the destination airports
airport_flight_dates_dest = df.groupby('Dest')['knownWeatherDateTime_UTC'].agg(['min', 'max']).reset_index()
# get the min out of both min dates, and the max out of both max dates, to get the date range for which we need weather data for each airport
airport_flight_dates = airport_flight_dates.merge(airport_flight_dates_dest, left_on='Origin', right_on='Dest', how='outer', suffixes=('_origin', '_dest'))
airport_flight_dates['min'] = airport_flight_dates[['min_origin', 'min_dest']].min(axis=1)
airport_flight_dates['max'] = airport_flight_dates[['max_origin', 'max_dest']].max(axis=1)
airport_flight_dates = airport_flight_dates[['Origin', 'min', 'max']].rename(columns={'Origin': 'iata_code'})

# subtract 1 day from the min date, and add 1 day to the max date, to get the date range for which we need weather data
airport_flight_dates['min'] = airport_flight_dates['min'] - pd.Timedelta(hours=5)
airport_flight_dates['max'] = airport_flight_dates['max'] + pd.Timedelta(hours=5)
# remove the timzone information from the min and max dates, as the meteostat library does not accept timezone information
airport_flight_dates['min'] = airport_flight_dates['min'].dt.tz_localize(None)
airport_flight_dates['max'] = airport_flight_dates['max'].dt.tz_localize(None)  


# merge the airport flight dates with the airports weather dataset to get the date range for which we need weather data for each airport
airport_weather_dates = airport_flight_dates.merge(airports_weather, on='iata_code', how='inner') 
# check if the length of all three datasets is the same, if not there are some airports for which we do not have weather data, and we need to drop them from the flights dataset
print(f"Length of airport_flight_dates: {len(airport_flight_dates)}")
print(f"Length of airports_weather: {len(airports_weather)}")
print(f"Length of airport_weather_dates: {len(airport_weather_dates)}")


weather_params = ["temp","prcp","wspd"]
weather_data = pd.DataFrame()

print("Getting weather data for each airport, this may take a while...")
def fill_weather_data(data, airport, date_range_length):
    data_length = len(data)
    # if the data is less than 95% complete, we try to fill it up with the data from the other near stations 1-3
    if data_length < date_range_length:
        print("filling up")
        # get data from another station
        for i in range(1,4):
            if airport[f'closest_station_{i}'] and airport[f'closest_station_{i}_distance'] < 50_000:
                data1 = ms.hourly(airport[f'closest_station_{i}'], airport['min'], airport['max'], parameters=weather_params)
                data1 = data1.fetch()
                if data1 is None or data1.empty:
                    continue
                data1["airport"] = airport["iata_code"]
                data1["timestamp"] = data1.index
                # match the data from the two stations and fill the missing rows in the first dataset with the values from the second dataset
                data = data.merge(data1, on=["timestamp", "airport"], how="outer", suffixes=("", "_1"))
                # fill every value on the first dataset with the value from the second dataset if it is missing in the first dataset
                for param in weather_params:
                    data[param] = data[param].fillna(data[f"{param}_1"])
                    data = data.drop(columns=[f"{param}_1"])
                if len(data) < date_range_length:
                    break
        print('finished filling up')
    return data
total_added_rows = 0
for index, airport in airport_weather_dates.iterrows():
    data_range_length = (airport['max'] - airport['min']).total_seconds() / 3600
    iata_code = airport['iata_code']
    # base case: we have an aiport station, with weather data, which is mostly complete
    data = None
    if airport['airport_station'] is not np.nan and airport['airport_data_length_code'] is not np.nan:
        data = ms.hourly(airport['airport_station'], airport['min'], airport['max'], parameters=weather_params)
        data = data.fetch()
        if data is None or data.empty:
            data = pd.DataFrame(columns=["timestamp", "airport"]+weather_params)
        data["airport"] = airport["iata_code"]
        data["timestamp"] = data.index
        # get the length of the data, and if it is less than 80% of the date range, we fill it up with the data from the other near stations 1-3
        data= fill_weather_data(data, airport, data_range_length)
    else:
        data = pd.DataFrame(columns=["timestamp", "airport"]+weather_params)
        # if we do not have an airport station, we try to fill the data with the data from the other near stations 1-3
        data = fill_weather_data(data, airport, data_range_length)
    # handle mssing data and nans
    # check if the data is complete
    if len(data) <= data_range_length:
        # there are gaps in the data.
        print(f" {iata_code} is incomplete. ({index+1}/{len(airport_weather_dates)}) --- {len(data)} rows ")
        # get the range of the data
        previ_length = len(data)
        data_range = (airport['min'], airport['max'])
        # create a complete range of timestamps for the date range
        complete_range = pd.date_range(start=data_range[0], end=data_range[1], freq='h')
        # make it a dataframe        
        complete_range = pd.DataFrame(complete_range, columns=['timestamp'])
        complete_range['airport'] = airport['iata_code']
        # merge the complete range with the data to get the missing timestamps
        data = complete_range.merge(data, on=['timestamp', 'airport'], how='outer')
        new_length = len(data)
        # print(f"added {new_length - previ_length} rows to the data for airport {iata_code}")
        total_added_rows += new_length - previ_length
    else: 
        print(f"{iata_code} is complete. ({index+1}/{len(airport_weather_dates)}) --- {len(data)} rows")
    # for all nans try using the previous value, if that is not also nan
    for param in weather_params:
        # try filling single nans with the previous value, if that is not also nan, we leave it as nan
        data[param] = data[param].ffill(limit=2)
    # hinterfragen wenn mit mehr daten gearbeite wird.
    # check if more than 50% of the values in the prcp column are missing, if so we fill them with 0, as it is more likely that there was no precipitation than that the data is missing
    
    if data['prcp'].isna().sum() / len(data) > 0.5:
        data['prcp'] = data['prcp'].fillna(-1)
    else:
        data['prcp'] = data['prcp'].fillna(0)
    # same for the wspd
    if data['wspd'].isna().sum() / len(data) > 0.5:
        data['wspd'] = data['wspd'].fillna(-1)
    else:
        data['wspd'] = data['wspd'].fillna(0)
    data['temp'] = data['temp'].fillna(0)
    weather_data = pd.concat([weather_data, data], ignore_index=True)
    
    # print(f"Got weather data for airport {iata_code} ({index+1}/{len(airport_weather_dates)}) --- {len(data)} rows")
print(f"Total rows added: {total_added_rows}")
print(len(df))
display(weather_data)


In [ ]:
df = df.merge(weather_data, left_on=["Origin", "knownWeatherDateTime_UTC"], right_on=["airport", "timestamp"], how="left", suffixes=("", "_DEP"))
df = df.merge(weather_data, left_on=["Dest", "knownWeatherDateTime_UTC"], right_on=["airport", "timestamp"], how="left", suffixes=("", "_ARR"))

display(df)

In [ ]:
# get all the rows where the timestamp_Dep or timestamp_ARR is missing
missing_arr = df[df['timestamp_ARR'].isna()]
missing_dep = df[df['timestamp'].isna()]
# union of both missing datasets
missing = pd.concat([missing_arr, missing_dep])
# throw out duplicates
missing = missing.drop_duplicates()
display(missing)
# drop the rows where the timestamp_Dep is missing, as we can not use them for training
df = df[~df['timestamp_ARR'].isna()]
df = df[~df['timestamp'].isna()]

# drop the temporary collumns
df = df.drop(columns=['knownWeatherDateTime_UTC', 'timestamp', 'timestamp_ARR','airport','airport_ARR'])

# convert the temp, prcp and wspd columns to numeric, as they are currently object due to the nans
df['temp'] = pd.to_numeric(df['temp'], errors='coerce')
df['prcp'] = pd.to_numeric(df['prcp'], errors='coerce')
df['wspd'] = pd.to_numeric(df['wspd'], errors='coerce')
df['temp_ARR'] = pd.to_numeric(df['temp_ARR'], errors='coerce')
df['prcp_ARR'] = pd.to_numeric(df['prcp_ARR'], errors='coerce')
df['wspd_ARR'] = pd.to_numeric(df['wspd_ARR'], errors='coerce')


display(df)

In [ ]:
# drop unessary columns 
# Timezone information, as we have already merged the weather data and know the timezone of the departure and arrival airports
# timestamps that are not schedulad departure and sheduled arrival 
# other things that are not useful for training
orther_cols =['CRSDepDateTime_UTC']

df = df.drop(columns=orther_cols)
# convert sheduled serv
df ['CRSDepDateTime'] = df['CRSDepDateTime'].dt.hour * 60 + df['CRSDepDateTime'].dt.minute
df ['CRSArrDateTime'] = df['CRSArrDateTime'].dt.hour * 60 + df['CRSArrDateTime'].dt.minute
display(df)

In [ ]:
# list of categrcal columns to be One-Hot Encoded
# cat_One_Hot =['Reporting_Airline','type_DEP','type_ARR','most_common_surface_DEP','most_common_surface_ARR']
cat_One_Hot =['Reporting_Airline','Origin','Dest']

# scale the numeric columns, except for the target column
scal_params = ['CRSDepDateTime', 'CRSArrDateTime', 'temp','prcp','wspd','temp_ARR','prcp_ARR','wspd_ARR','TurnaroundTime','CRSElapsedTime','Distance']

continus_cols =[]
# convert CRSDepDateTime and CRSArrDateTime to minutes since midnight


target = 'ArrDelayMinutes'
# split the Data into train and test set


In [ ]:
# One Hot Encode the categorical columns
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
encoder = OneHotEncoder(handle_unknown="ignore")
encoded_carriers = encoder.fit_transform(df[cat_One_Hot])
encoded_carriers_df = pd.DataFrame(encoded_carriers.toarray(), columns=encoder.get_feature_names_out(cat_One_Hot))
# join the dataframes together
df = pd.concat([df.drop(columns=cat_One_Hot), encoded_carriers_df], axis=1, join="inner")


# scale the numeric columns
scaler = MinMaxScaler()
df[scal_params] = scaler.fit_transform(df[scal_params])

In [ ]:
# perform a time based split of the data, using the CRSDepDateTime column, to split the data into a train set with flights before 2023-01-01 and a test set with flights after 2023-01-01
test_threshold = pd.to_datetime("2011-08-01").timestamp()
train = df[df['CRSDepDateTime'] < test_threshold]
test = df[df['CRSDepDateTime'] >= test_threshold]
X_train = train.drop(columns=[target])
y_train = train[target]
X_test = test.drop(columns=[target])
y_test = test[target]
print(f"Length of train set: {len(train)}")
print(f"Length of test set: {len(test)}")


In [ ]:
# perform a time sensitve hyperparmeter tuning and cross validation





In [ ]:
# function for performing a time sensitve hyperparmeter tuning and cross validation, using the TimeSeriesSplit and RandomizedSearchCV from sklearn, with a given model and parameter grid
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
def time_sensitive_hyperparameter_tuning(model, param_grid, X_train, y_train):
    splitter = TimeSeriesSplit(n_splits=5)
    random_search = RandomizedSearchCV(model, param_grid, cv=splitter, n_iter=10, scoring='R2', n_jobs=-1)
    random_search.fit(X_train, y_train)
    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

In [ ]:
# test the model 
def test_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    print(f"Mean Squared Error: {mse}")
    print(f"R^2 Score: {r2}")
    return mse, r2

# fucntion to plot Predicted vs Actual values
def plot_predictions(Y_test, predictions, title=f"Predicted vs Actual {target}", Metrics:dict=None):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=Y_test, y=predictions, alpha=0.5)
    plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--')  # Line for perfect predictions
    plt.xlabel(f"Actual {target}")
    plt.ylabel(f"Predicted {target}")
    # cut off the axis at 0 and 450 to focus on the most relevant range of values
    plt.xlim(0, 450)
    plt.ylim(0, 450)
    plt.title(title)
    if Metrics:
        plt.text(0.05, 0.95, f"MSE: {Metrics['MSE']:.2f}\nR²: {Metrics['R2']:.2f}", transform=plt.gca().transAxes, verticalalignment='top')
    plt.show()

In [ ]:
# pipline 
def pipeline(model, param_grid, X_train, y_train, X_test, y_test):
    best_model, best_params, best_score = time_sensitive_hyperparameter_tuning(model, param_grid, X_train, y_train)
    print(f"Best Hyperparameters: {best_params}")
    print(f"Best Cross-Validation R² Score: {best_score:.4f}")
    mse, r2 = test_model(best_model, X_test, y_test)
    plot_predictions(y_test, best_model.predict(X_test), title=f"Predicted vs Actual {target} with Best Hyperparameters", Metrics={"MSE": mse, "R2": r2})

def test_on_train_subset(model, X_train_subset, Y_train_subset):
    predictions = model.fit(X_train_subset, Y_train_subset).predict(X_train_subset)
    mse = mean_squared_error(Y_train_subset, predictions)
    r2 = r2_score(Y_train_subset, predictions)
    print(f"Train Subset - Mean Squared Error: {mse:.2f}")
    print(f"Train Subset - R^2 Score: {r2:.2f}")
    return model, predictions, mse, r2

In [ ]:
# function for tree or forest models to plot the feature importance
def plot_feature_importance(model, feature_names, top_n=50):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    plt.figure(figsize=(10, 10))
    sns.barplot(x=importances[indices], y=np.array(feature_names)[indices])
    plt.title("Feature Importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.show()

In [ ]:
# try out a simple linear regression model
from sklearn.linear_model import LinearRegression
model = LinearRegression()
param_grid = {
    'fit_intercept': [True, False],
    'normalize': [True, False]
}
pipeline(model, param_grid, X_train, y_train, X_test, y_test)

In [ ]:
# Descion Tree Regressor
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor(random_state=42, min_samples_leaf=50)
param_grid = {
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
pipeline(model, param_grid, X_train, y_train, X_test, y_test)

In [ ]:
# XGBoost Regressor
from xgboost import XGBRegressor
model = XGBRegressor(random_state=42, n_estimators=100, max_depth=25, learning_rate=0.1)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'learning_rate': [0.01, 0.1, 0.2]
}
pipeline(model, param_grid, X_train, y_train, X_test, y_test)
# find the 30 most important features for the XGBoost model
plot_feature_importance(model, X_train.columns)


In [ ]:
# adaboost regressor
from sklearn.ensemble import AdaBoostRegressor
model = AdaBoostRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
pipline(model)
# plot the feature importance of the adaboost model, top 50 features
plot_feature_importance(model, X_train2.columns)

In [ ]:
# random forest regressor
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42, n_estimators=100, max_depth=25)
pipline(model)
plot_feature_importance(model, X_train2.columns)
